# Week 6 · Notebook 2 — CNNs & Sequences: A Quick Tour
**CSCI 250 · Fall 2026**

Dense layers treat every input independently. Two specialized ideas add structure: **convolution** for images and **sequence models** for ordered data (the lineage that leads to the Transformer behind every LLM).

> No training here — just intuition you can run.

## 1. Convolution: a filter sliding over an image
A CNN learns small **filters** that slide across an image. Here we apply a fixed *vertical-edge* filter to a tiny synthetic image and see it light up the edge — exactly what a CNN's first layer learns to do.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# tiny 8x8 image: left half dark, right half bright (a vertical edge)
img = np.zeros((8, 8)); img[:, 4:] = 1.0

# a 3x3 vertical-edge detector (Sobel-like)
kernel = np.array([[-1, 0, 1],
                   [-2, 0, 2],
                   [-1, 0, 1]], dtype=float)

def convolve2d(x, k):
    kh, kw = k.shape
    out = np.zeros((x.shape[0]-kh+1, x.shape[1]-kw+1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = np.sum(x[i:i+kh, j:j+kw] * k)
    return out

feat = convolve2d(img, kernel)
fig, ax = plt.subplots(1, 2, figsize=(7, 3))
ax[0].imshow(img, cmap='gray'); ax[0].set_title('input image')
ax[1].imshow(np.abs(feat), cmap='magma'); ax[1].set_title('edge feature map')
plt.show()

The filter responds strongly right where the edge is. Stack many such filters across many layers and a CNN goes from edges → textures → objects. **Same neurons/weights, arranged to share parameters across the image.**

## 2. Sequences: predicting the next item
Sequence models read ordered data one step at a time (RNN/LSTM) or all at once (Transformer). The training objective is the heart of an LLM: **given what came before, predict the next item.** Here is that objective on a toy number sequence using a simple frequency table — no deep net needed to see the idea.

In [ ]:
seq = list('AABABBAABABBAABABB')   # a repeating pattern
from collections import defaultdict, Counter
nxt = defaultdict(Counter)
for a, b in zip(seq, seq[1:]):
    nxt[a][b] += 1

def predict_next(prev):
    counts = nxt[prev]
    total = sum(counts.values())
    return {k: round(v/total, 2) for k, v in counts.items()}

for c in 'AB':
    print(f"after '{c}'  -> next-item probabilities {predict_next(c)}")

Those probabilities are a baby version of an LLM's **softmax over the vocabulary**. A real Transformer conditions on the *whole* context with **attention** instead of just the previous character — but the goal (probability of the next token) is identical.

## 3. Optional: a real CNN layer in PyTorch
If torch is available, here is one true `Conv2d` layer applied to our image — the framework version of the convolution we coded above. Guarded so the notebook still runs without torch.

In [ ]:
try:
    import torch, torch.nn as nn
    conv = nn.Conv2d(1, 1, kernel_size=3, bias=False)
    with torch.no_grad():
        conv.weight.copy_(torch.tensor(kernel).float().view(1, 1, 3, 3))
    x = torch.tensor(img).float().view(1, 1, 8, 8)
    out = conv(x).detach().numpy()[0, 0]
    print('Conv2d output shape:', out.shape)
    plt.imshow(np.abs(out), cmap='magma'); plt.title('nn.Conv2d feature map'); plt.show()
except Exception as e:
    print('PyTorch not available — skipping (the NumPy version above already showed the idea).', e)

## Takeaway
CNNs and Transformers are not new math — they are the **same neurons, layers, activations, loss, and gradient descent** from Notebook 1, arranged for images and sequences. That is the foundation under the LLMs we use for the rest of CSCI 250.